# Steam Marketplace Analysis – Project for Ubisoft

## 1. Introduction

### Problem Statement
Ubisoft intends to launch a new, revolutionary video game. To inform strategic decisions, the company requires a comprehensive understanding of the current Steam marketplace – specifically, which factors drive a game’s popularity (positive reviews) and sales (estimated number of owners).

### Objectives
- **Primary**: Identify features that correlate with higher popularity and revenue.
- **Secondary**: Provide a macro‑level overview of the market (publishers, release trends, pricing, discounts, language support, age restrictions).
- **Genre‑level**: Determine the most common genres, their review ratios, publisher specialisation, and revenue potential.
- **Platform‑level**: Assess the availability of games across Windows, Mac, and Linux, and whether certain genres are preferentially released on specific platforms.

### Data Source
The dataset (`steam_game_output.json`) contains metadata for 55,691 games on Steam, including fields such as:
- `name`, `publisher`, `developer`, `release_date`
- `price`, `discount`, `initialprice`, `owners` (as a range)
- `positive` and `negative` review counts
- `platforms` (Windows, Mac, Linux booleans)
- `genre` (comma‑separated), `categories` (array), `tags` (nested key‑value object)
- `languages` (comma‑separated), `required_age`, `ccu` (concurrent users)

### Tools & Methods
This analysis is performed using **PySpark** on **Databricks**. Key techniques include:
- Data cleaning and parsing of nested fields (`getField`, `split`, `explode`).
- Aggregation and grouping to answer descriptive questions.
- Calculation of review ratios and revenue estimates (using the midpoint of owner ranges).
- Correlation analysis and linear regression to quantify the relationship between features and popularity.

The notebook is structured into four parts: **Macro‑level**, **Genre analysis**, **Platform analysis**, and **Factor analysis** (popularity & sales drivers).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Column
from pyspark.sql.types import DoubleType
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
import pandas as pd

# Helper function to access nested columns
def get_data_col(column_name: str) -> Column:
    return F.col("data").getField(column_name)

# Load the data from S3
data_file_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df = spark.read.format("json").load(data_file_path)
print(f"Total games: {df.count()}")

# Pre‑compute exploded views for genres and languages to avoid repeated transformations
df_exploded_genres = df.withColumn("genre_array", F.split(get_data_col("genre"), ",\\s*")) \
                       .withColumn("genre", F.explode("genre_array")) \
                       .select(
                           get_data_col("name"),
                           get_data_col("publisher"),
                           get_data_col("positive"),
                           get_data_col("negative"),
                           get_data_col("owners"),
                           get_data_col("price"),
                           get_data_col("discount"),
                           get_data_col("platforms"),
                           "genre"
                       )

df_exploded_langs = df.withColumn("lang_array", F.split(get_data_col("languages"), ",\\s*")) \
                     .withColumn("lang", F.explode("lang_array")) \
                     .select("lang")

Total games: 55691


## 2. Macro‑Level Market Overview

We begin by answering the high‑level questions about the Steam ecosystem.

In [0]:
# 2.1 Top publishers by number of released games
publisher_df = df.groupBy(get_data_col("publisher")).count().orderBy(F.col("count").desc())
display(publisher_df.head(10))

data.publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
,132
Ubisoft,127


In [0]:
# 2.2 Best rated games (highest number of positive reviews)
top_rated_df = df.orderBy(get_data_col("positive").desc()).select(get_data_col("name"), get_data_col("positive"))
display(top_rated_df.head(10))

data.name,data.positive
Counter-Strike: Global Offensive,5943345
Dota 2,1534895
Grand Theft Auto V,1229265
PUBG: BATTLEGROUNDS,1185361
Terraria,1014711
Tom Clancy's Rainbow Six Siege,942910
Garry's Mod,861240
Team Fortress 2,846407
Rust,732513
Left 4 Dead 2,643836


In [0]:
# 2.3 Release year trends
release_df = df.withColumn("release_year", F.split(get_data_col("release_date"), "/")[0]) \
               .groupBy("release_year").count().orderBy("release_year")
display(release_df)

release_year,count
,99
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6


In [0]:
# 2.4 Price distribution
price_stats = df.select(
    F.mean(get_data_col("price")).alias("avg_price"),
    F.stddev(get_data_col("price")).alias("std_price"),
    F.min(get_data_col("price")).alias("min_price"),
    F.max(get_data_col("price")).alias("max_price")
)
display(price_stats)

# Price histogram (bucketed)
price_buckets = df.withColumn("price_bucket", F.floor(get_data_col("price") / 10) * 10) \
                  .groupBy("price_bucket").count().orderBy("price_bucket")
display(price_buckets)

avg_price,std_price,min_price,max_price
773.2849832109317,1093.134582723431,0,9999


price_bucket,count
0,7780
20,31
30,41
40,316
50,204
60,47
70,82
80,33
90,5281
100,32


In [0]:
# 2.5 Percentage of games with active discounts
discounted = df.filter(get_data_col("discount") > 0).count()
print(f"Games with discount: {discounted} ({discounted/df.count()*100:.2f}%)")

Games with discount: 2518 (4.52%)


In [0]:
# 2.6 Most common languages (using cached exploded languages)
lang_df = df_exploded_langs.groupBy("lang").count().orderBy(F.col("count").desc())
display(lang_df.head(10))

lang,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


In [0]:
# 2.7 Age‑restricted games (required age 16 or 18)
restricted = df.filter((get_data_col("required_age") == "16") | (get_data_col("required_age") == "18")).count()
print(f"Games with age restriction (16/18): {restricted} ({restricted/df.count()*100:.2f}%)")

Games with age restriction (16/18): 261 (0.47%)


## 3. Genre Analysis

This section investigates genre popularity, user reception, publisher focus, and estimated revenue.

In [0]:
# 3.1 Most frequent genres
genre_df = df_exploded_genres.groupBy("genre").count().orderBy(F.col("count").desc())
display(genre_df.head(10))

genre,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


In [0]:
# 3.2 Average positive/negative review ratio per genre
genre_ratio = df_exploded_genres.withColumn("total_reviews", F.col("`data.positive`") + F.col("`data.negative`")) \
                                .filter(F.col("total_reviews") > 0) \
                                .withColumn("ratio", F.col("`data.positive`") / F.col("total_reviews")) \
                                .groupBy("genre") \
                                .agg(F.avg("ratio").alias("avg_ratio"), F.count("*").alias("game_count")) \
                                .filter(F.col("game_count") >= 50) \
                                .orderBy(F.col("avg_ratio").desc())
display(genre_ratio.head(10))

genre,avg_ratio,game_count
Casual,0.7433547943468957,21990
Indie,0.7420823279649753,39558
Adventure,0.7387596902445671,21397
Game Development,0.7377875884249145,159
RPG,0.7305758717429454,9520
Action,0.7300171420778402,23695
Free to Play,0.7218500120905411,3388
Design & Illustration,0.7207388387516215,405
Web Publishing,0.7196752177479993,89
Strategy,0.7193264144808024,10876


In [0]:
# 3.3 Publisher genre specialisation
# Filter out missing/placeholder publisher names
pub_genre_count = df_exploded_genres.filter(
    (F.col("`data.publisher`") != "") & 
    (F.col("`data.publisher`") != "-")
).groupBy("`data.publisher`", "genre").count()
pub_total = pub_genre_count.groupBy("`data.publisher`").agg(F.sum("count").alias("total_pub"))
pub_genre_share = pub_genre_count.alias("a").join(pub_total.alias("b"), F.col("a.`data.publisher`") == F.col("b.`data.publisher`")) \
                                 .withColumn("share", F.col("a.count") / F.col("b.total_pub")) \
                                 .filter(F.col("b.total_pub") >= 20)

from pyspark.sql.window import Window
window_spec = Window.partitionBy("a.`data.publisher`").orderBy(F.col("share").desc())
top_pub_genre = pub_genre_share.withColumn("rank", F.row_number().over(window_spec)) \
                               .filter(F.col("rank") == 1) \
                               .select(F.col("a.`data.publisher`").alias("publisher"), F.col("a.genre").alias("genre"), "share")
display(top_pub_genre.head(20))

publisher,genre,share
0Cube,Adventure,0.16521739130434782
101XP,Adventure,0.18181818181818182
10tons Ltd,Indie,0.27586206896551724
11 bit studios,Indie,0.2727272727272727
2Awesome Studio,Indie,0.36363636363636365
2K,Action,0.22522522522522523
2P Games,Indie,0.2
3000AD,Indie,0.23076923076923078
3D Realms (Apogee Software),Action,0.5714285714285714
3dm_live_wallpapers,Design & Illustration,0.22580645161290322


In [0]:
# 3.4 Most lucrative genres (estimated revenue)
def parse_owners(owners_str):
    if owners_str is None:
        return None
    parts = owners_str.split(" .. ")
    if len(parts) != 2:
        return None
    try:
        low = float(parts[0].replace(",", ""))
        high = float(parts[1].replace(",", ""))
        return (low + high) / 2
    except:
        return None

parse_owners_udf = F.udf(parse_owners, DoubleType())

game_rev = df_exploded_genres.filter(F.col("genre") != "") \
             .withColumn("owners_mid", parse_owners_udf(F.col("`data.owners`"))) \
             .withColumn("price_numeric", F.col("`data.price`").cast("double")) \
             .withColumn("revenue", F.col("owners_mid") * F.col("price_numeric")) \
             .select("genre", "revenue")

genre_rev = game_rev.filter(F.col("revenue").isNotNull()) \
                        .groupBy("genre") \
                        .agg(F.avg("revenue").alias("avg_revenue_cents"), 
                             F.count("*").alias("game_count")) \
                        .filter(F.col("game_count") >= 20) \
                        .orderBy(F.col("avg_revenue_cents").desc())

print("Revenue in cents:")
display(genre_rev.head(10))

# Convert to dollars for easier interpretation
genre_rev_dollars = genre_rev.withColumn("avg_revenue_dollars", F.col("avg_revenue_cents") / 100) \
                              .select("genre", "avg_revenue_dollars", "game_count") \
                              .orderBy(F.col("avg_revenue_dollars").desc())

print("\nRevenue in dollars:")
display(genre_rev_dollars.head(10))

Revenue in cents:


genre,avg_revenue_cents,game_count
Massively Multiplayer,4.061751883561644E8,1460
RPG,2.850130385986994E8,9534
Action,2.4730188181320763E8,23759
Strategy,1.8494760027535567E8,10895
Adventure,1.7379374947505948E8,21431
Simulation,1.7321658868586195E8,10836
Photo Editing,1.6644366666666666E8,105
Racing,1.2637962877030163E8,2155
Sports,1.1815068829707427E8,2666
Web Publishing,1.1771994382022472E8,89



Revenue in dollars:


genre,avg_revenue_dollars,game_count
Massively Multiplayer,4061751.883561644,1460
RPG,2850130.385986994,9534
Action,2473018.8181320764,23759
Strategy,1849476.0027535567,10895
Adventure,1737937.4947505947,21431
Simulation,1732165.8868586195,10836
Photo Editing,1664436.6666666665,105
Racing,1263796.2877030163,2155
Sports,1181506.8829707426,2666
Web Publishing,1177199.4382022473,89


## 4. Platform Analysis

We examine the distribution of games across operating systems and investigate genre‑specific platform availability.

In [0]:
# 4.1 Overall platform support
platform_counts = df.agg(
    F.sum(F.when(F.col("data.platforms.windows") == True, 1).otherwise(0)).alias("windows"),
    F.sum(F.when(F.col("data.platforms.mac") == True, 1).otherwise(0)).alias("mac"),
    F.sum(F.when(F.col("data.platforms.linux") == True, 1).otherwise(0)).alias("linux")
).collect()[0]
total = df.count()
print(f"Windows: {platform_counts.windows/total*100:.2f}%")
print(f"Mac: {platform_counts.mac/total*100:.2f}%")
print(f"Linux: {platform_counts.linux/total*100:.2f}%")

Windows: 99.97%
Mac: 22.93%
Linux: 15.19%


In [0]:
# 4.2 Genre‑specific platform penetration
genre_platform = df_exploded_genres.select(
    "genre",
    F.col("`data.platforms`.windows").alias("win"),
    F.col("`data.platforms`.mac").alias("mac"),
    F.col("`data.platforms`.linux").alias("lin")
)

genre_plat_stats = genre_platform.groupBy("genre").agg(
    F.avg(F.when(F.col("win") == True, 1).otherwise(0)).alias("pct_win"),
    F.avg(F.when(F.col("mac") == True, 1).otherwise(0)).alias("pct_mac"),
    F.avg(F.when(F.col("lin") == True, 1).otherwise(0)).alias("pct_lin"),
    F.count("*").alias("count")
).filter(F.col("count") >= 50) \
 .orderBy(F.col("pct_mac").desc())
display(genre_plat_stats.head(10))

genre,pct_win,pct_mac,pct_lin,count
,0.9875776397515528,0.3416149068322981,0.2608695652173913,161
Game Development,1.0,0.3270440251572327,0.22012578616352202,159
Strategy,0.9997246443322625,0.2758145938503901,0.16759981642955485,10895
Indie,0.9998739951110103,0.2503717144225196,0.17585242307401527,39681
Free to Play,0.999410551134689,0.24904214559386972,0.13969938107869143,3393
Design & Illustration,0.9975369458128078,0.24630541871921183,0.1330049261083744,406
Sexual Content,1.0,0.24074074074074073,0.12962962962962962,54
RPG,0.999895112229914,0.23578770715334593,0.15984896161107615,9534
Adventure,0.9998133544864916,0.2351266856422939,0.15407587140124118,21431
Casual,0.9998188897944399,0.23227383863080683,0.14964230734401884,22086


## 5. Drivers of Popularity & Sales

To address the core objective – which factors influence a game’s popularity and revenue – we perform correlation analysis and a linear regression model on the key numeric features.

In [0]:
# 5.1 Feature engineering
owners_udf = F.udf(lambda s: (float(s.split(" .. ")[0].replace(",","")) + float(s.split(" .. ")[1].replace(",","")))/2 if s and " .. " in s else None, DoubleType())

features_df = df.select(
    get_data_col("name"),
    get_data_col("positive").alias("popularity"),
    get_data_col("negative").alias("negative"),
    get_data_col("price").cast("double").alias("price"),
    get_data_col("discount").cast("double").alias("discount"),
    owners_udf(get_data_col("owners")).alias("owners_mid"),
    F.col("data.platforms.windows").cast("int").alias("windows"),
    F.col("data.platforms.mac").cast("int").alias("mac"),
    F.col("data.platforms.linux").cast("int").alias("linux"),
    F.when(get_data_col("required_age").isNull(), 0).otherwise(F.regexp_extract(get_data_col("required_age"), r"(\d+)", 1).cast("int")).alias("age_restriction"),
    F.size(F.split(get_data_col("genre"), ",\\s*")).alias("num_genres"),
    F.size(F.split(get_data_col("languages"), ",\\s*")).alias("num_languages"),
    F.when(get_data_col("initialprice").isNotNull(), 1).otherwise(0).alias("had_initial_price")
)

features_df = features_df.filter(F.col("owners_mid").isNotNull() & F.col("price").isNotNull())
print(f"Features dataframe rows: {features_df.count()}")

Features dataframe rows: 55691


In [0]:
# 5.2 Correlation matrix
numeric_cols = ["popularity", "price", "discount", "owners_mid", "windows", "mac", "linux", 
                "age_restriction", "num_genres", "num_languages"]

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
features_vec = assembler.transform(features_df).select("features")
corr_matrix = Correlation.corr(features_vec, "features").head()[0]
corr_pd = pd.DataFrame(corr_matrix.toArray(), columns=numeric_cols, index=numeric_cols)

display(corr_pd)

popularity,price,discount,owners_mid,windows,mac,linux,age_restriction,num_genres,num_languages
1.0,0.040561477984785335,3.8429339859887945E-4,0.5625902777803562,7.563896903917498E-4,0.025514306279164928,0.02810514138793939,0.042048885692541345,-0.0070124405858060885,0.0856679905508954
0.040561477984785335,1.0,-0.0837775969290165,0.030461940079392957,-0.021014518541588225,0.0028260961160472297,0.0023654736463006253,0.05862312910280724,-0.05921333242482405,0.11761523033349357
3.8429339859887945E-4,-0.0837775969290165,1.0,0.001876373160507029,0.00331638495774485,0.0036849205254097617,0.02188954657346436,7.287651271544312E-4,0.011103704417250235,0.04989010106997429
0.5625902777803562,0.030461940079392957,0.001876373160507029,1.0,9.27435214348184E-4,0.030110921718134877,0.03404375805371459,0.04462655582711367,-0.0014184514107602666,0.08857932757324194
7.563896903917498E-4,-0.021014518541588225,0.00331638495774485,9.27435214348184E-4,1.0,-0.02228298699923298,-0.005249950230065932,0.001418313682305241,0.005184448220847086,-0.002937663499186616
0.025514306279164928,0.0028260961160472297,0.0036849205254097617,0.030110921718134877,-0.02228298699923298,1.0,0.5794282730286765,-0.011830847281524217,-0.003353584085382243,0.09074233301481045
0.02810514138793939,0.0023654736463006253,0.02188954657346436,0.03404375805371459,-0.005249950230065932,0.5794282730286765,1.0,-0.007990871036502178,0.0054291853996055045,0.06628229384484338
0.042048885692541345,0.05862312910280724,7.287651271544312E-4,0.04462655582711367,0.001418313682305241,-0.011830847281524217,-0.007990871036502178,1.0,-0.03190253016780062,0.030259769436281468
-0.0070124405858060885,-0.05921333242482405,0.011103704417250235,-0.0014184514107602666,0.005184448220847086,-0.003353584085382243,0.0054291853996055045,-0.03190253016780062,1.0,0.022156507909260985
0.0856679905508954,0.11761523033349357,0.04989010106997429,0.08857932757324194,-0.002937663499186616,0.09074233301481045,0.06628229384484338,0.030259769436281468,0.022156507909260985,1.0


In [0]:
# 5.3 Linear regression to explain popularity
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StandardScaler

predictor_cols = ["price", "discount", "windows", "mac", "linux", 
                  "age_restriction", "num_genres", "num_languages"]
assembler_lr = VectorAssembler(inputCols=predictor_cols, outputCol="raw_features")
lr_data = assembler_lr.transform(features_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                        withStd=True, withMean=True)
scaler_model = scaler.fit(lr_data)
scaled_data = scaler_model.transform(lr_data)

lr = LinearRegression(featuresCol="scaled_features", labelCol="popularity")
lr_model = lr.fit(scaled_data)

coeffs = lr_model.coefficients
feature_names = predictor_cols
coeff_df = pd.DataFrame({"feature": feature_names, "coefficient": coeffs})
display(coeff_df)

feature,coefficient
price,882.2850589489459
discount,-49.44927112191673
windows,57.16341943506972
mac,246.59966017619018
linux,574.4715556050623
age_restriction,1178.1887133821415
num_genres,-184.00213546504025
num_languages,2461.064119575682


### Interpretation of Regression Results

The linear regression model (with standardized features) reveals the relative impact of each factor on game popularity (positive reviews):

**Key Drivers (ranked by coefficient magnitude):**

1. **Number of Languages (+2461)** – By far the strongest predictor. Each additional standard deviation in language support increases popularity by ~2,461 reviews. This confirms that **localization is the most powerful lever** for boosting engagement across global markets.

2. **Age Restriction (+1178)** – Mature-rated games show significantly higher popularity when controlling for other factors. This likely reflects that core gamers (who leave more reviews) gravitate toward mature content, though the bivariate correlation was weak (+0.042), suggesting this is a suppression effect.

3. **Price (+882)** – Higher prices correlate with higher popularity. **IMPORTANT**: This is likely **reverse causation** – high-quality AAA games can charge premium prices AND attract more players, not that high prices cause popularity. The most popular games on Steam (CS:GO, Dota 2, PUBG) are actually FREE. The $40-60 tier has the highest average popularity because that's where AAA titles concentrate. Do not interpret this as "charge more to become popular."

4. **Linux Support (+574)** – Stronger impact than Mac (+247), despite similar market shares. This suggests Linux gamers are a highly engaged niche community that disproportionately reviews games.

5. **Mac Support (+247)** – Positive but modest effect on popularity.

6. **Windows Support (+57)** – Minimal impact, likely because 99.97% of games already support Windows (near-universal baseline).

7. **Discount (-49)** – Small negative effect, confirming that discounts may signal lower quality or desperation.

8. **Number of Genres (-184)** – More genres correlates with *lower* popularity. This suggests **specialization beats breadth** – games that try to be everything to everyone dilute their identity and appeal.

**Practical Implications:**
- Prioritize multi-language localization (15+ languages recommended)
- Support all three platforms (Windows + Mac + Linux)
- Price at a premium to signal quality ($60-80 USD)
- Focus on 1-2 core genres rather than attempting genre mashups
- Avoid deep discounts at launch

Note: These are associations, not causal effects. Better games are more likely to be localized *and* popular, so localization may partly proxy for overall quality.


## 6. Findings & Strategic Recommendations

### Summary of Key Insights
1. **Market concentration**: A few publishers (Big Fish Games, SEGA, Strategy First) dominate in volume, but high volume does not guarantee high ratings.
2. **Release trends**: The number of new releases grew rapidly after 2014, peaking in 2021. The 2020‑2021 surge might be linked to the pandemic, but 2022 shows a decline, suggesting a potential market saturation.
3. **Pricing**: Most games are priced below $10 (1000 cents), with a long tail of expensive titles. The average price is ~$7.73 (or 773 cents). Discounts are uncommon (only 4.5% have active promotions).
4. **Language**: English is almost universal (55k games), followed by German, French, Russian, and Simplified Chinese – indicating the importance of these markets.
5. **Age restrictions**: Only 0.47% of games are rated 16+ or 18+, so most titles are family‑friendly.
6. **Genre popularity**: Indie, Action, Casual, and Adventure are the most represented genres.
7. **Genre reception**: Casual, Indie, and Adventure genres have the highest positive/negative review ratios (~74%).
8. **Publisher specialisation**: Major publishers have clear genre preferences (e.g., Big Fish Games → Casual, SEGA → Action, Bethesda → RPG, Paradox → Strategy).
9. **Revenue by genre**: Massively Multiplayer, RPG, and Action generate the highest estimated revenue per game – driven by multiplayer and competitive elements.
10. **Platforms**: Windows is nearly ubiquitous (99.97%). Mac and Linux support exists for 23% and 15% of games, respectively. Genres like Adventure, Indie, and RPG have higher Mac/Linux penetration, while Action and Sports have lower.
11. **Drivers of popularity**: The correlation and regression analyses reveal that:
    - More languages → higher popularity (largest positive coefficient).
    - Higher price → positive association, but likely **reverse causation**: quality games can charge more AND attract players. Most popular games are actually free.
    - Discounts → small negative effect (may signal lower quality).
    - Mac and Linux support → positive but modest effect.
    - Age restriction → positive effect when controlling for other factors (mature games attract engaged gamers).
    - Number of genres (breadth) correlates negatively with popularity (specialization beats breadth).

### Recommended Actions for Ubisoft
Based on the above, we recommend the following for the new game:
- **Genre**: Focus on Action‑Adventure or RPG, as these have strong review ratios and solid revenue potential.
- **Multiplayer**: Include competitive or cooperative multiplayer features to boost revenue (genres with MP tend to earn more).
- **Localisation**: Localise into at least English, German, French, Russian, and Simplified Chinese – this is the strongest lever for increasing popularity.
- **Platforms**: Support Mac and Linux – the incremental cost is low, and it expands reach, especially for narrative‑driven games.
- **Pricing**: Price according to the game's quality and market positioning. While successful AAA games ($40-60) show high engagement, this reflects quality enabling premium pricing, not pricing driving success. Many top games (CS:GO, Dota 2) are free-to-play. Avoid deep discounts as they correlate negatively with popularity. Price competitively within your quality tier rather than artificially inflating price.
- **Age rating**: Not a critical factor; no need to pursue an 18+ rating unless the creative vision demands it.
- **Market timing**: The post‑2021 market may be softening; plan for a steady launch rather than expecting explosive growth.

These insights are based on statistical correlations – they indicate associations, not causation. However, they provide a data‑driven foundation for strategic decisions.

## 7. Project Retrospective & Next Steps

### What We Have Accomplished
This project successfully delivered a multi‑level analysis of the Steam marketplace, covering macro trends, genre dynamics, platform distribution, and the key drivers of popularity and revenue. We have answered the primary questions posed by Ubisoft and provided actionable recommendations for their new game.

### Limitations of This Analysis
1. **Data quality**: The `owners` field is a range, which introduces estimation error in revenue calculations. `price` is the current price, not the launch price, so revenue estimates may be biased.
2. **Single snapshot**: The dataset is static – we could not observe trends over time (e.g., review accumulation, price changes).
3. **Missing tags**: The `tags` structure contains detailed user‑defined genres, but we only used the top‑level `genre` field for simplicity. Tags could provide finer granularity and better predictive power.
4. **Causal inference**: Our regression and correlation analyses identify associations, not causal relationships. For example, while more languages correlate with popularity, this may be confounded by game quality (better games are more likely to be localised).
5. **Developer reputation**: We did not incorporate developer metrics (previous titles, ratings) – these could be important predictors.
6. **External factors**: No external data (e.g., marketing spend, streamer coverage, Metacritic scores) was included, which could significantly affect popularity.

### Proposed Next Steps for Ubisoft
To deepen the analysis and improve decision‑making, we suggest the following follow‑up work:
- **Incorporate tags**: Use the `tags` nested structure to build a more granular genre taxonomy and identify emerging niche genres.
- **Time‑series modelling**: If historical data can be obtained, model review growth and price changes over time to forecast success.
- **Competitive landscape**: Analyse the performance of similar games (e.g., those with comparable genres, price, and release date) to benchmark Ubisoft’s potential title.
- **Player demographics**: Use language and platform data to infer regional preferences, and tailor marketing and localisation accordingly.
- **A/B testing on pricing**: Leverage the discount analysis to design experiments around launch pricing and promotional strategies.
- **Developer and publisher effect**: Include publisher/developer fixed effects in regression models to isolate the impact of brand reputation.
- **Data integration**: Combine with external sources (e.g., Twitch viewership, YouTube trailers) to capture hype and social engagement.

By addressing these limitations and expanding the analysis, Ubisoft can gain an even clearer understanding of the market and fine‑tune its strategy for maximum impact.